# Duka Electronics | H1 2026 Board Report

## Storyboard — agree the story before plotting

**Audience:** The Duka Electronics Board and CEO; non-technical readers reviewing the first half of 2026, often on a phone.

**Decision:** Where should leadership focus commercial attention and resources in the second half of 2026?

**Big Idea:** H1 revenue is concentrated in Nairobi and laptops, but sales softened after April; protect the strongest engine while rebuilding momentum across the rest of the portfolio.

**SCR (Situation–Complication–Resolution):**
1. **Situation:** The business generated KES 71.8 million from 1,000 orders across five branches in H1.
2. **Complication:** Nairobi and laptops dominate revenue, while monthly revenue fell from an April peak to June; this leaves the business exposed to concentration and weakening momentum.
3. **Resolution:** Defend Nairobi/laptop execution and run targeted H2 recovery tests in lower-revenue branches and channels, reviewing monthly results before scaling.

*All figures describe the provided synthetic Week 3 sales dataset; revenue is quantity × unit price. Currency is Kenyan shillings (KES).*

## Board snapshot

| H1 revenue | Orders | Top branch | Top product |
| ---: | ---: | --- | --- |
| **KES 71.8M** | **1,000** | **Nairobi · KES 26.1M** | **Laptop · KES 41.5M** |

## Report page

The report is exported at `reports/duka_board_report.png`. Each panel has an insight-led headline, clear units, one accent highlight, direct values and a source note.

In [ ]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("duka_sales.csv")
REPORT_DIR = Path("reports")
FIGURE_DIR = Path("figures")
REPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
sales = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month_start"] = sales["order_date"].dt.to_period("M").dt.to_timestamp()

monthly_revenue = sales.groupby("month_start")["revenue"].sum().sort_index()
branch_revenue = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)
product_revenue = sales.groupby("product")["revenue"].sum().sort_values(ascending=False)
channel_revenue = sales.groupby("channel")["revenue"].sum().sort_values(ascending=False)
total_revenue = int(sales["revenue"].sum())
peak_month = monthly_revenue.idxmax()
trough_month = monthly_revenue.idxmin()
month_drop_pct = (monthly_revenue.iloc[-1] / monthly_revenue.iloc[monthly_revenue.index.get_loc(peak_month)] - 1) * 100
nairobi_share = branch_revenue.loc["Nairobi"] / total_revenue * 100
laptop_share = product_revenue.loc["Laptop"] / total_revenue * 100

sns.set_theme(style="whitegrid", context="notebook")
INK = "#182B3A"
ACCENT = "#D95F3D"
BLUE = "#3D6F8E"
PALE = "#DCE5E9"
SOURCE = "Source: Duka Electronics sales dataset, Jan–Jun 2026; revenue = quantity × unit price."

def format_millions(value, _position=None):
    return f"{value / 1_000_000:.0f}M"

def add_source(ax):
    ax.text(0.0, -0.22, SOURCE, transform=ax.transAxes, ha="left", va="top", fontsize=7, color="#586873")

fig, axes = plt.subplots(2, 2, figsize=(13.33, 7.5), constrained_layout=False)
fig.patch.set_facecolor("#F7F9FA")
fig.suptitle("DUKA ELECTRONICS  |  H1 2026 BOARD REPORT", x=0.07, y=0.975, ha="left", fontsize=19, fontweight="bold", color=INK)
fig.text(0.07, 0.925, f"KES {total_revenue / 1e6:.1f}M revenue  ·  {len(sales):,} orders  ·  A clear H2 priority: protect the core, restore momentum", ha="left", fontsize=10, color="#435665")

# 1. Monthly trend: use one line and direct point labels; title is computed from peak and H1 slide.
ax = axes[0, 0]
month_labels = [calendar.month_abbr[d.month] for d in monthly_revenue.index]
trend_title = f"Revenue peaked in {calendar.month_name[peak_month.month]} at KES {monthly_revenue.max() / 1e6:.1f}M, then fell {abs(month_drop_pct):.1f}% by June"
ax.set_title(trend_title, loc="left", fontsize=10, fontweight="bold", color=INK, pad=10)
ax.plot(month_labels, monthly_revenue.values / 1e6, color=BLUE, linewidth=2.5, marker="o", markersize=5)
ax.scatter([month_labels[monthly_revenue.values.argmax()]], [monthly_revenue.max() / 1e6], color=ACCENT, s=48, zorder=3)
for label, value in zip(month_labels, monthly_revenue.values / 1e6):
    ax.annotate(f"{value:.1f}", (label, value), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8, color=INK)
ax.set_ylabel("Revenue (KES millions)", fontsize=8)
ax.set_xlabel("Month, 2026", fontsize=8)
ax.grid(axis="x", visible=False)
ax.spines[["top", "right"]].set_visible(False)
add_source(ax)

# 2. Branch comparison: rank bars and highlight Nairobi only.
ax = axes[0, 1]
branch_title = f"Nairobi leads with KES {branch_revenue.iloc[0] / 1e6:.1f}M ({nairobi_share:.0f}% of H1 revenue)"
ax.set_title(branch_title, loc="left", fontsize=10, fontweight="bold", color=INK, pad=10)
branch_values = branch_revenue / 1e6
branch_colors = [ACCENT if name == "Nairobi" else PALE for name in branch_values.index]
bars = ax.barh(branch_values.index[::-1], branch_values.values[::-1], color=branch_colors[::-1], height=0.62)
for bar, value in zip(bars, branch_values.values[::-1]):
    ax.text(value + 0.35, bar.get_y() + bar.get_height() / 2, f"{value:.1f}", va="center", fontsize=8, color=INK)
ax.set_xlabel("Revenue (KES millions)", fontsize=8)
ax.set_ylabel("Branch", fontsize=8)
ax.set_xlim(0, branch_values.max() * 1.25)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
add_source(ax)

# 3. Product portfolio composition: sorted bars with the revenue-leading laptop highlighted.
ax = axes[1, 0]
product_title = f"Laptops deliver KES {product_revenue.iloc[0] / 1e6:.1f}M — {laptop_share:.0f}% of total sales"
ax.set_title(product_title, loc="left", fontsize=10, fontweight="bold", color=INK, pad=10)
product_values = product_revenue / 1e6
product_colors = [ACCENT if name == "Laptop" else PALE for name in product_values.index]
bars = ax.barh(product_values.index[::-1], product_values.values[::-1], color=product_colors[::-1], height=0.62)
for bar, value in zip(bars, product_values.values[::-1]):
    ax.text(value + 0.35, bar.get_y() + bar.get_height() / 2, f"{value:.1f}", va="center", fontsize=8, color=INK)
ax.set_xlabel("Revenue (KES millions)", fontsize=8)
ax.set_ylabel("Product", fontsize=8)
ax.set_xlim(0, product_values.max() * 1.25)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
add_source(ax)

# 4. Channel mix: direct-labelled share and totals make the acquisition mix legible.
ax = axes[1, 1]
channel_share = channel_revenue / total_revenue * 100
ax.set_title(f"In-store remains the largest channel at {channel_share.iloc[0]:.0f}% of revenue", loc="left", fontsize=10, fontweight="bold", color=INK, pad=10)
channel_values = channel_revenue / 1e6
bars = ax.barh(channel_values.index[::-1], channel_values.values[::-1], color=[PALE, BLUE, PALE][::-1], height=0.62)
for bar, name, value in zip(bars, channel_values.index[::-1], channel_values.values[::-1]):
    ax.text(value + 0.3, bar.get_y() + bar.get_height() / 2, f"KES {value:.1f}M  ·  {channel_share[name]:.0f}%", va="center", fontsize=8, color=INK)
ax.set_xlabel("Revenue (KES millions)", fontsize=8)
ax.set_ylabel("Channel", fontsize=8)
ax.set_xlim(0, channel_values.max() * 1.5)
ax.grid(axis="y", visible=False)
ax.spines[["top", "right", "left"]].set_visible(False)
add_source(ax)

fig.text(0.07, 0.035, "BOARD ACTION  |  Defend Nairobi + laptop execution; test focused recovery plays in other branches and monitor monthly.", fontsize=10, fontweight="bold", color=INK)
fig.subplots_adjust(left=0.09, right=0.96, top=0.86, bottom=0.13, hspace=0.56, wspace=0.32)
report_path = REPORT_DIR / "duka_board_report.png"
fig.savefig(report_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"Saved one-page board report: {report_path}")
print(f"H1 revenue: KES {total_revenue:,}; peak month: {calendar.month_name[peak_month.month]}; June vs peak: {month_drop_pct:.1f}%")

## Final report

![One-page Duka Electronics H1 2026 board report](reports/duka_board_report.png)

## Before-and-after makeover

**Chart:** Monthly revenue, the same measure and data in both versions.

**Changes:** The before version uses default styling, unsignalled month bars and no explanatory headline. The after version uses a chronological line to reveal direction, calculates the peak-to-June change in its title, highlights April with the single accent colour, labels values directly, adds KES units and a source note, and removes unnecessary borders/gridlines. The values and scope are unchanged.

In [ ]:
# Before: default, unranked visual encoding for the same monthly revenue series.
before_fig, before_ax = plt.subplots(figsize=(7, 4))
before_ax.bar([calendar.month_abbr[d.month] for d in monthly_revenue.index], monthly_revenue.values / 1e6)
before_ax.set_title("Revenue by Month")
before_ax.set_ylabel("Revenue")
before_fig.savefig(FIGURE_DIR / "before.png", dpi=160, bbox_inches="tight")
plt.show()

# After: match the polished trend treatment used on the board page.
after_fig, after_ax = plt.subplots(figsize=(7, 4))
after_labels = [calendar.month_abbr[d.month] for d in monthly_revenue.index]
after_ax.plot(after_labels, monthly_revenue.values / 1e6, color=BLUE, marker="o", linewidth=2.5)
after_ax.scatter([after_labels[monthly_revenue.values.argmax()]], [monthly_revenue.max() / 1e6], color=ACCENT, s=55, zorder=3)
for label, value in zip(after_labels, monthly_revenue.values / 1e6):
    after_ax.annotate(f"KES {value:.1f}M", (label, value), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
after_ax.set_title(f"April peak; revenue down {abs(month_drop_pct):.1f}% by June", loc="left", fontweight="bold")
after_ax.set_ylabel("Revenue (KES millions)")
after_ax.set_xlabel("Month, 2026")
after_ax.grid(axis="x", visible=False)
after_ax.spines[["top", "right"]].set_visible(False)
after_ax.text(0, -0.2, SOURCE, transform=after_ax.transAxes, fontsize=7, color="#586873")
after_fig.tight_layout()
after_fig.savefig(FIGURE_DIR / "after.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved figures/before.png and figures/after.png")

### Comparison

**Before**

![Before makeover](figures/before.png)

**After**

![After makeover](figures/after.png)

## Reading the evidence

The strongest revenue concentration is the Nairobi branch (KES 26.1M; 36% of H1) and laptops (KES 41.5M; 58%). April is the monthly high at KES 13.5M; June is KES 10.9M, 19.3% below April. In-store contributes KES 41.0M (57%), Online KES 22.8M (32%), and Phone order KES 8.0M (11%). These descriptive totals support focused H2 testing, not causal claims or a full profitability decision; margins, costs, stock availability and post-June data are not included.